In [1]:
import sys
from pathlib import Path

# Resuelve el problema de las ubicaciones de los scripts
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data import DataRepository
from src.preprocessing import PreprocessingPipeline
from src.eda import StatisticsAnalyzer, OutlierDetector

# Tema visual por defecto del proyecto.
sns.set_theme(style=config.SEABORN_THEME, palette=config.PLOT_PALETTE)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 200)

# Carga de la información
repo = DataRepository()
dataframe = repo.load()
repo.metadata.to_dict()

{'source': 'Secretariado Ejecutivo del Sistema Nacional de Seguridad Pública (SESNP), mediante datos.gob.mx',
 'title': 'Incidencia Delictiva Estatal',
 'extraction_date': '2026-06-01',
 'path': '/Users/wallsified/Documents/UNAM/2026-2/Almacenes y Mineria de Datos/proyecto-final-aymdd/data/INM_estatal_dic25.csv',
 'url': 'https://www.datos.gob.mx/dataset/incidencia_delictiva/resource/d9b2792a-33a2-4ea8-8527-210d9e99de5e',
 'encoding': 'utf-8'}

In [2]:
repo.dictionary()

,columna,tipo,descripcion
0,anio,int64,Año de registro de las averiguaciones previas y/o carpetas de investigación.
1,clave_ent,int64,"Clave de la entidad, según el Marco Geoestadístico Nacional (MGN) del Instituto Nacional de Geografía y Estadística (INEGI)."
2,entidad,str,Entidad federativa de registro de las averiguaciones previas y/o carpetas de investigación.
3,bien_juridico_afectado,str,"Primera clasificación de los delitos en las averiguaciones previas y/o carpetas de investigación. (Patrimonio, vida, libertad, etc.)"
4,tipo_delito,str,"Segunda clasificación de los delitos. (Falsedad, Lesiones, Robo, etc.)"
5,subtipo_delito,str,"Tercera clasificación de los delitos. Subcategorías específicas de cada tipo de delito. (Lesiones dolosas, etc. Puede ser repetido con la anterior)"
6,modalidad,str,"Cuarta clasificación de los delitos. Forma en la que se comete el delito. (Con violencia, sin violencia, etc.)"
7,mes,str,Mes de registro de las averiguaciones previas y/o carpetas de investigación.
8,fecha,str,Fecha de registro de las averiguaciones previas y/o carpetas de investigación.
9,incidencia_delictiva,int64,Incidencia delictiva del Fuero Común (Número absoluto de presuntos delitos registrados)


In [3]:
cleaning_pipeline = PreprocessingPipeline(drop_columns=["anio", "clave_ent", "mes", "entidad_federativa"])
clean_dataset = cleaning_pipeline.run(dataframe)
# reorganizamos las columnas para tener la fecha como columna inicial del dataset
clean_dataset = clean_dataset.iloc[:, [5,0,1,2,3,4,6]]
cleaning_pipeline.save(clean_dataset)

[1] Eliminando columnas declaradas en drop_columns
 Eliminadas: ['anio', 'clave_ent', 'mes', 'entidad_federativa']
[2] Convirtiendo '{self._date_col}' a datetime
   OK=413,952 / fallidos=0
 Rango: 2015-01-01 → 2025-12-01
[3] Eliminando filas con nulos en críticas
Filas: 413,952 -> 413,952 (eliminadas: 0)
[4] Eliminando duplicados exactos
Filas: 413,952 -> 413,952 (eliminadas: 0)
CSV limpio guardado en:
 /Users/wallsified/Documents/UNAM/2026-2/Almacenes y Mineria de Datos/proyecto-final-aymdd/data/processed/INM_estatal_dic25_clean.csv
413,952 filas x 7 columnas, 


In [4]:
repo = DataRepository('../data/processed/INM_estatal_dic25_clean.csv')
dataframe = repo.load()
stats = StatisticsAnalyzer(dataframe)
stats.numeric_stats()

,columna,Promedio,Mediana,Desviacíon Estándar,Valor Mínimo,Q1,Q3,Valor Máximo,range
0,incidencia_delictiva,52.499106,2.0,204.518424,0.0,0.0,26.0,9968.0,9968.0


In [5]:
stats.categorical_stats()

,columna,valores únicos,moda,frecuencia modal
0,fecha,132,2015-04-01,3136
1,entidad,32,Aguascalientes,12936
2,bien_juridico_afectado,7,El patrimonio,177408
3,tipo_delito,40,Robo,152064
4,subtipo_delito,55,Robo de maquinaria,25344
5,modalidad,59,Con violencia,50688


In [9]:
stats.frequency_table('tipo_delito', top_k=50)

,tipo_delito,count
0,Robo,152064
1,Homicidio,38016
2,Lesiones,38016
3,Secuestro,21120
4,Feminicidio,16896
5,Abuso de confianza,4224
6,Daño a la propiedad,4224
7,Despojo,4224
8,Extorsión,4224
9,Fraude,4224


- IQR La incidencia delictiva está fuertemente sesgada a la derecha (CDMX y Edomex disparan los valores frente a estados pequeños). El z-score asume normalidad y los propios extremos inflan media y desviación, así que reporta resultados sesgados; IQR es robusto a colas largas y coincide con lo que dibuja el boxplot.
- Tratamiento: marcar, no eliminar. En estos datos los outliers no son errores de captura: son señales reales. Eliminarlos distorsionaría el análisis. La clase ofrece además ``winsorize`` (recorte a los límites) y ``drop`` (eliminación de filas) para cuando se justifique.

In [6]:
det = OutlierDetector(dataframe, columns=["incidencia_delictiva"])
det.summary()

,columna,n_total,min,max,lower,upper,n_outliers,pct_outliers
0,incidencia_delictiva,413952,0.0,9968.0,-39.0,65.0,63128,15.250077


In [9]:
det.flag()

,fecha,entidad,bien_juridico_afectado,tipo_delito,subtipo_delito,modalidad,incidencia_delictiva,incidencia_delictiva_is_outlier
0,2015-04-01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,22,False
1,2015-08-01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,40,False
2,2015-12-01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,26,False
3,2015-01-01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,41,False
4,2015-02-01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,33,False
...,...,...,...,...,...,...,...,...
413947,2025-03-01,Zacatecas,Otros bienes jurídicos afectados (del fuero común),Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,184,True
413948,2025-05-01,Zacatecas,Otros bienes jurídicos afectados (del fuero común),Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,98,True
413949,2025-11-01,Zacatecas,Otros bienes jurídicos afectados (del fuero común),Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,92,True
413950,2025-10-01,Zacatecas,Otros bienes jurídicos afectados (del fuero común),Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,99,True
